In [1]:
!pip install pandas numpy plotly gradio openpyxl

**Load Dataset**

In [3]:
import pandas as pd

df = pd.read_excel("/content/Accounts-Payable.xlsx")

df.head()

,APID,Vendor,InvoiceDate,DueDate,Amount,Currency,Status,PaidDate,Terms
0,AP10000,BluePrints,2024-12-16,2025-01-15,344.25,EUR,Open,NaT,Net 30
1,AP10001,BluePrints,2024-05-18,2024-06-17,3298.38,CAD,Partial,NaT,Net 30
2,AP10002,Fast Travel,2024-02-25,2024-03-26,1023.43,GBP,Open,NaT,Net 30
3,AP10003,ABC Supplies,2024-01-08,2024-02-07,4433.05,USD,Partial,NaT,Net 30
4,AP10004,ABC Supplies,2024-05-23,2024-06-22,4345.14,GBP,Partial,NaT,Net 30


**Data Cleaning**

In [4]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['DueDate'] = pd.to_datetime(df['DueDate'])
df['PaidDate'] = pd.to_datetime(df['PaidDate'], errors='coerce')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   APID         800 non-null    object        
 1   Vendor       800 non-null    object        
 2   InvoiceDate  800 non-null    datetime64[ns]
 3   DueDate      800 non-null    datetime64[ns]
 4   Amount       800 non-null    float64       
 5   Currency     800 non-null    object        
 6   Status       800 non-null    object        
 7   PaidDate     257 non-null    datetime64[ns]
 8   Terms        800 non-null    object        
dtypes: datetime64[ns](3), float64(1), object(5)
memory usage: 56.4+ KB


**Create Exception Rules
Overdue Invoice**

In [5]:
today = pd.Timestamp.today()

df['Overdue'] = (
    (df['Status'] != 'Paid') &
    (df['DueDate'] < today)
)

**High Value Invoice**

In [6]:
df['HighValue'] = df['Amount'] > 5000

**Duplicate Invoice Detection**

In [7]:
df['Duplicate'] = df.duplicated(
    subset=['Vendor','Amount','InvoiceDate'],
    keep=False
)

**Missing Payment**

In [8]:
df['MissingPayment'] = (
    (df['Status'] == 'Paid') &
    (df['PaidDate'].isna())
)

**Exception Classification**

In [9]:
def classify(row):

    if row['Duplicate']:
        return "Duplicate Invoice"

    elif row['Overdue']:
        return "Overdue Invoice"

    elif row['MissingPayment']:
        return "Missing Payment Date"

    elif row['HighValue']:
        return "High Value Invoice"

    else:
        return "Normal"

df['ExceptionType'] = df.apply(classify, axis=1)

df[['APID','Vendor','ExceptionType']].head()

,APID,Vendor,ExceptionType
0,AP10000,BluePrints,Overdue Invoice
1,AP10001,BluePrints,Overdue Invoice
2,AP10002,Fast Travel,Overdue Invoice
3,AP10003,ABC Supplies,Overdue Invoice
4,AP10004,ABC Supplies,Overdue Invoice


**Priority Engine**

In [10]:
def priority(row):

    if row['ExceptionType']=="Duplicate Invoice":
        return "High"

    elif row['ExceptionType']=="Overdue Invoice":
        return "High"

    elif row['ExceptionType']=="High Value Invoice":
        return "Medium"

    else:
        return "Low"

df['Priority'] = df.apply(priority, axis=1)

**AI Resolution Recommendation**

In [11]:
def recommendation(exception):

    mapping = {

        "Duplicate Invoice":
        "Review invoice and verify duplicate payment risk.",

        "Overdue Invoice":
        "Escalate to Accounts Payable Manager.",

        "Missing Payment Date":
        "Validate payment records and update system.",

        "High Value Invoice":
        "Require approval from Finance Head.",

        "Normal":
        "No action required."
    }

    return mapping.get(exception)

df['Recommendation'] = (
    df['ExceptionType']
    .apply(recommendation)
)

**Dashboard**

In [12]:
import plotly.express as px

fig = px.pie(
    df,
    names='ExceptionType',
    title='Invoice Exceptions'
)

fig.show()

**Vendor Risk Analysis**

In [13]:
vendor_risk = (
    df.groupby('Vendor')
    .size()
    .reset_index(name='Invoices')
)

vendor_risk.sort_values(
    'Invoices',
    ascending=False
).head()

,Vendor,Invoices
1,BluePrints,182
3,Global Office,165
0,ABC Supplies,157
4,TechMart,157
2,Fast Travel,139


**Aging Analysis**

In [14]:
df['DaysOverdue'] = (
    today - df['DueDate']
).dt.days

df['DaysOverdue'] = (
    df['DaysOverdue']
    .clip(lower=0)
)

**Gradio Web App**

In [15]:
import gradio as gr

def analyze():

    return df.head(50)

gr.Interface(
    fn=analyze,
    inputs=None,
    outputs="dataframe",
    title="AP Invoice Exception System"
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fd0234aa66adc6825f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Export Results**

In [16]:
df.to_excel(
    "Invoice_Exception_Report.xlsx",
    index=False
)